# Side check: the Kepler post

The post claims a new planet: a **0.24% dip every 13.77 days** on **KIC 5689121**, found by pulling apart blended Kepler
pixels. Before believing a discovery, do what this whole project trains you to do: **check it against what is already known.**

Read `docs/kepler_post_check.md` alongside this notebook.

In [ ]:
import io
import numpy as np
import pandas as pd
import requests
from astropy.coordinates import SkyCoord
from astroquery.vizier import Vizier

POST_PERIOD_DAYS = 13.77
POST_DEPTH = 0.0024            # 0.24 %

## 1. The star in the post, and its neighbours in the Kepler Input Catalog

In [ ]:
catalog = Vizier(columns=["KIC", "RAJ2000", "DEJ2000", "kepmag", "Teff", "logg"], row_limit=5)

def kic_star(kic):
    return catalog.query_constraints(catalog="V/133/kic", KIC=str(kic))[0][0]

post_star = kic_star(5689121)
print(post_star)
print("Kepler magnitude", post_star["kepmag"], "| temperature", post_star["Teff"], "K -> a cool red dwarf, as the post says")

## 2. Is there a known planet with that period nearby?
The NASA Exoplanet Archive lists every Kepler Object of Interest (KOI). A neighbouring catalogue number, KIC 5689351, is KOI-505:

In [ ]:
query = ("select kepoi_name, kepler_name, kepid, koi_disposition, koi_period, koi_time0bk, koi_depth, koi_kepmag "
         "from cumulative where kepid = 5689351")
response = requests.get("https://exoplanetarchive.ipac.caltech.edu/TAP/sync", params={"query": query, "format": "csv"}, timeout=120)
koi_505 = pd.read_csv(io.StringIO(response.text))
koi_505

In [ ]:
planet = koi_505[koi_505["kepoi_name"] == "K00505.01"].iloc[0]
neighbour = kic_star(5689351)
separation = SkyCoord(post_star["RAJ2000"], post_star["DEJ2000"], unit="deg").separation(
             SkyCoord(neighbour["RAJ2000"], neighbour["DEJ2000"], unit="deg")).arcsec
print(f"{planet['kepler_name']}: CONFIRMED planet, period {planet['koi_period']:.4f} d  vs the post's {POST_PERIOD_DAYS} d")
print(f"its star is {separation:.0f} arcsec away = {separation / 3.98:.0f} Kepler pixels, "
      f"and {10 ** ((post_star['kepmag'] - neighbour['kepmag']) / 2.5):.1f}x brighter")

## 3. Could the period match be a coincidence?
Rough estimate: suppose an unrelated signal's period is equally likely anywhere in log-period between 1 and 100 days.
The chance of landing within ±0.01 d of one of this system's 5 known periods:

In [ ]:
tolerance = 0.01
chance_per_period = (2 * tolerance / koi_505["koi_period"]) / np.log(100)
print(f"chance of a coincidental match with any of the 5 periods: {chance_per_period.sum():.1e}")

About **3 in 1,000** (1 in 350). Not impossible, but a shared origin is the better bet, and it must be ruled out before claiming a new planet.

## 4. But can leaked light make a 0.24% dip? (The dilution argument)
If a fraction of the bright star's light lands in the faint star's aperture, the observed dip is the bright star's depth,
**diluted** by the faint star's own light. It can only get *shallower*, never deeper:

In [ ]:
true_depth = planet["koi_depth"] * 1e-6          # ppm -> fraction
brightness_ratio = 10 ** ((post_star["kepmag"] - neighbour["kepmag"]) / 2.5)
for leaked_fraction in [0.01, 0.1, 0.5, 1.0, 10.0]:
    leaked = leaked_fraction * brightness_ratio   # leaked light in units of the faint star's flux
    observed = true_depth * leaked / (1 + leaked)
    print(f"leaked fraction of the bright star's light {leaked_fraction:5}: dip {observed:.4%}")
print(f"maximum possible from simple blending: {true_depth:.4%}   the post reports: {POST_DEPTH:.2%}")

## 5. Verdict (honest, not final)
- **For contamination:** a confirmed planet with the *same period*, 4.4' away; a coincidence has odds of about 1 in 350.
- **Against simple blending:** at 66 pixels, a star's light barely reaches the other aperture, and blended light can't
  produce a dip 3.5× deeper than the source transit.
- **What could reconcile them:** effects that aren't simple blending, such as electronic **crosstalk** between CCD
  outputs, or a **baseline error** in custom pixel photometry. Over-subtracting the background shrinks the faint star's
  measured flux, which inflates any *relative* dip.

**The decisive test:** compare the post's transit *times* with Kepler-169 e's ephemeris (period and mid-transit time
`koi_time0bk`). If the dips line up, it's Kepler-169 e leaking in; if they're offset, it's something else and deserves a closer look.

In [ ]:
t0 = planet["koi_time0bk"]                        # BKJD = BJD - 2454833
print(f"Kepler-169 e transits at BKJD = {t0:.5f} + n x {planet['koi_period']:.5f}")
print("first few:", np.round(t0 + np.arange(5) * planet["koi_period"], 3))

## Lessons for our asteroid search
1. **Cross-match before you celebrate.** Our pipeline checks every tracklet against SkyBoT; the post's author needed the KOI list.
2. **Ask what else could produce the signal**, and test each explanation with numbers (here: the dilution limit).
3. **A coincidence estimate** turns "looks similar" into a number (here about 1 in 350).
4. **Name the decisive test** instead of declaring victory.